# Module 1 — pilot Drive → Colab → Drive

Chạy trên **Google Colab**. Chỉ xử lý BTS, BTW, BPC; `YEARS = None` nghĩa là tất cả năm có trong ba mã này. Nguồn OCR và PNG chỉ được đọc qua Drive API; mỗi báo cáo tải tạm vào máy ảo Colab rồi xóa. Artifact và checkpoint ghi dưới `multi_agent_financial/module_01_pilot/`. Code lấy từ GitHub. Chạy lại sẽ bỏ qua report có completion marker hợp lệ cùng nguồn, config và commit code.

Đây là nhánh OCR có sẵn. Quality chạy trên toàn bộ trang; normalization v1 chỉ hỗ trợ bảng kết quả kinh doanh có layout rõ; C1 chạy cho các bảng chuẩn hóa được và không kết luận pháp lý khi chưa có registry quy định đã duyệt. Không tự publish vào C2 hay nâng trạng thái document lên `ready`.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
try:
    from googleapiclient.discovery import build
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'google-api-python-client'], check=True)
    from googleapiclient.discovery import build
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive_service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
print('Drive API ready')

## Lấy mã nguồn từ Git
Lần đầu clone; những lần sau pull fast-forward. Mỗi completion marker lưu commit SHA để resume không dùng nhầm kết quả của code khác.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/Mutil_Agent_Financial_Report')
url = 'https://github.com/VuongQuocAn/Mutil_Agent_Financial_Report.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Chạy pilot và tiếp tục khi chạy lại
Giữ `YEARS = None` để chạy mọi năm của BTS, BTW, BPC. Đặt tuple năm, ví dụ `('2025',)`, khi cần thử một năm trước. Kết quả lỗi được báo theo từng báo cáo; những báo cáo khác tiếp tục chạy.

In [ ]:
from financial_multiagent.ingestion.drive_colab_runner import PilotConfig, run_pilot
YEARS = None
result = run_pilot(drive_service, PilotConfig(years=YEARS))
for item in result:
    print(item['ticker'], item['year'], item['status'], item.get('drive_folder_id', ''), item.get('error', ''))
assert all(item['status'] != 'failed' for item in result), 'Review failed reports above; rerun after fixing the cause'